<a href="https://colab.research.google.com/github/carlogr-13/Dynamic_Allostery_PLMs/blob/main/perturbation_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.2. Perfil perturbación epistática y propagación distal

Magnitud del recableado inducido por la mutación I150A mediante análisis diferencial de tensores de acoplamiento, evaluando tanto el impacto directo sobre el andamiaje hidrofóbico como la transmisión de señal epistática a larga distancia.

*   Matriz maestra de impacto diferencial
*   Mapeo de propagación distal



## 4.2.1. Impacto directo en el andamiaje hidrofóbico basal

In [6]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# MONTAJE Y NAVEGACIÓN A LA CARPETA DE TOPOLOGÍA
drive.mount('/content/drive')
ruta_topologia = '/content/drive/MyDrive/TFG_notebooks/Data_PKA/quantitative_metrics/'
os.chdir(ruta_topologia)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import pandas as pd
import numpy as np
import os
from google.colab import files  # <-- LÍNEA AÑADIDA PARA SOLUCIONAR EL ERROR

# 1. Lista de residuos del núcleo hidrofóbico (C-spine, R-spine, anclaje aF)
target_residues = [57, 70, 95, 106, 128, 164, 172, 173, 174, 185, 220, 222, 227, 231]
master_df = None

print("Construyendo la Matriz Maestra de Perturbación Direccional...")

# 2. Bucle de extracción y cálculo de Delta JSD
for res in target_residues:
    wt_file = f"DirectedSensitivity_{res}_WT.csv"
    mut_file = f"DirectedSensitivity_{res}_I150A.csv"

    if os.path.exists(wt_file) and os.path.exists(mut_file):
        df_wt = pd.read_csv(wt_file)
        df_mut = pd.read_csv(mut_file)

        # Fusión estricta por el identificador del residuo
        df_merged = pd.merge(df_wt, df_mut, on="Residue_PDB", suffixes=('_WT', '_Mut'))

        # Cálculo de la Fluctuación Direccional (Delta JSD)
        col_name = f'Delta_JSD_{res}'
        df_merged[col_name] = df_merged['Epistatic_Coupling_JSD_Mut'] - df_merged['Epistatic_Coupling_JSD_WT']

        # Extraer datos relevantes y limpiar nombres
        df_delta = df_merged[['Residue_PDB', 'Amino_Acid_WT', col_name]].copy()
        df_delta.rename(columns={'Amino_Acid_WT': 'Amino_Acid'}, inplace=True)

        # Integración en la matriz maestra (Outer join)
        if master_df is None:
            master_df = df_delta
        else:
            master_df = pd.merge(master_df, df_delta, on=['Residue_PDB', 'Amino_Acid'], how='outer')
    else:
        print(f" [!] Advertencia: Faltan archivos para el residuo emisor {res}")

# 3. Limpieza de datos
master_df.fillna(0, inplace=True)

# 4. PREGUNTA A: ANÁLISIS DE SUMIDEROS (Suma de filas - axis=1)
delta_cols = [col for col in master_df.columns if 'Delta_JSD' in col]
master_df['Divergencia_Receptiva_Acumulada'] = master_df[delta_cols].abs().sum(axis=1)

# Ordenar por el impacto total recibido
master_df.sort_values(by='Divergencia_Receptiva_Acumulada', ascending=False, inplace=True)

# Guardar y Descargar
output_name = "Master_Delta_JSD_Matrix_Completa.csv"
master_df.to_csv(output_name, index=False)
files.download(output_name)

# 5. Reporte de Resultados
print("\n" + "="*70)
print(" TOP 10 SUMIDEROS DE INFORMACIÓN (Fuga de la señal del núcleo)")
print(" (Residuos de la quinasa que reciben el mayor impacto de la mutación)")
print("="*70)
print(master_df[['Residue_PDB', 'Amino_Acid', 'Divergencia_Receptiva_Acumulada']].head(10).to_string(index=False))

Construyendo la Matriz Maestra de Perturbación Direccional...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 TOP 10 SUMIDEROS DE INFORMACIÓN (Fuga de la señal del núcleo)
 (Residuos de la quinasa que reciben el mayor impacto de la mutación)
 Residue_PDB Amino_Acid  Divergencia_Receptiva_Acumulada
         150          I                         0.020660
         113          N                         0.008640
         235          Y                         0.006968
         153          T                         0.005632
         142          H                         0.004680
         182          V                         0.003903
         180          I                         0.003067
          97          A                         0.002877
         128          M                         0.002773
         176          Q                         0.002643


In [9]:
# PREGUNTA B: INESTABILIDAD DE EMISIÓN (Suma de columnas - axis=0)

print("="*70)
print(" INESTABILIDAD EPISTÁTICA DE EMISIÓN (Nodos del núcleo alterados)")
print(" (Cuánto ha cambiado la firma epistática que emite cada residuo)")
print("="*70)

# Sumamos el valor absoluto de cada columna Delta
emisiones = master_df[delta_cols].abs().sum(axis=0).sort_values(ascending=False)

print(f"{'Emisor (Núcleo)':<20} | {'Inestabilidad Total Generada (Σ |ΔJSD|)':<40}")
print("-" * 70)

for col, valor in emisiones.items():
    residuo_emisor = col.replace('Delta_JSD_', '')

    # Clasificación semántica basada en bibliografía (Taylor & Kornev)
    if int(residuo_emisor) in [95, 106, 164, 185]:
        etiqueta = "(R-spine)"
    elif int(residuo_emisor) in [57, 70, 128, 172, 173, 174]:
        etiqueta = "(C-spine)"
    else:
        etiqueta = "(Anclaje aF)"

    print(f"Residuo {residuo_emisor:<3} {etiqueta:<10} | {valor:.5f}")

 INESTABILIDAD EPISTÁTICA DE EMISIÓN (Nodos del núcleo alterados)
 (Cuánto ha cambiado la firma epistática que emite cada residuo)
Emisor (Núcleo)      | Inestabilidad Total Generada (Σ |ΔJSD|) 
----------------------------------------------------------------------
Residuo 172 (C-spine)  | 0.03066
Residuo 106 (R-spine)  | 0.01613
Residuo 227 (Anclaje aF) | 0.01485
Residuo 95  (R-spine)  | 0.01109
Residuo 222 (Anclaje aF) | 0.00618
Residuo 231 (Anclaje aF) | 0.00617
Residuo 174 (C-spine)  | 0.00578
Residuo 185 (R-spine)  | 0.00561
Residuo 128 (C-spine)  | 0.00504
Residuo 164 (R-spine)  | 0.00481
Residuo 57  (C-spine)  | 0.00375
Residuo 173 (C-spine)  | 0.00292
Residuo 220 (Anclaje aF) | 0.00227
Residuo 70  (C-spine)  | 0.00167


In [10]:
# PREGUNTA C: CROSS-TALK INTERNO DEL NÚCLEO HIDROFÓBICO

print("="*70)
print(" MAPA DE CROSS-TALK INTRA-NÚCLEO (Desacoplamientos y Refuerzos)")
print(" (Alteración en la comunicación directa entre las espinas)")
print("="*70)

# 1. Filtramos para quedarnos solo con la interacción entre los residuos diana
df_intra = master_df[master_df['Residue_PDB'].isin(target_residues)].copy()
df_intra.set_index('Residue_PDB', inplace=True)

# 2. Recopilación matricial de los emparejamientos
interacciones = []

for emisor in target_residues:
    col_emisor = f'Delta_JSD_{emisor}'
    if col_emisor in df_intra.columns:
        for receptor in target_residues:
            if receptor in df_intra.index and emisor != receptor:
                delta_val = df_intra.at[receptor, col_emisor]
                interacciones.append({
                    'Emisor': emisor,
                    'Receptor': receptor,
                    'Delta_JSD': delta_val,
                    'Impacto_Absoluto': abs(delta_val)
                })

# 3. Ordenamiento por mayor impacto absoluto
df_cross = pd.DataFrame(interacciones)
df_cross.sort_values(by='Impacto_Absoluto', ascending=False, inplace=True)

# 4. Reporte del Top 15 de rutas internas fracturadas o anómalas
print(f"{'Emisor':<8} | {'Receptor':<8} | {'Delta JSD':<10} | {'Diagnóstico Interno'}")
print("-" * 65)

for _, row in df_cross.head(15).iterrows():
    e = int(row['Emisor'])
    r = int(row['Receptor'])
    delta = row['Delta_JSD']

    if delta > 0:
        dinamica = "Acoplamiento anómalo de estrés (+)"
    elif delta < 0:
        dinamica = "Desacoplamiento / Fractura (-)"
    else:
        dinamica = "Estabilidad (Sin cambios)"

    print(f"Res {e:<4} | Res {r:<4} | {delta:>10.5f} | {dinamica}")

 MAPA DE CROSS-TALK INTRA-NÚCLEO (Desacoplamientos y Refuerzos)
 (Alteración en la comunicación directa entre las espinas)
Emisor   | Receptor | Delta JSD  | Diagnóstico Interno
-----------------------------------------------------------------
Res 172  | Res 128  |   -0.00164 | Desacoplamiento / Fractura (-)
Res 172  | Res 227  |    0.00066 | Acoplamiento anómalo de estrés (+)
Res 174  | Res 172  |   -0.00061 | Desacoplamiento / Fractura (-)
Res 227  | Res 231  |    0.00053 | Acoplamiento anómalo de estrés (+)
Res 231  | Res 128  |   -0.00045 | Desacoplamiento / Fractura (-)
Res 106  | Res 128  |    0.00024 | Acoplamiento anómalo de estrés (+)
Res 231  | Res 227  |    0.00022 | Acoplamiento anómalo de estrés (+)
Res 173  | Res 174  |    0.00021 | Acoplamiento anómalo de estrés (+)
Res 128  | Res 172  |   -0.00016 | Desacoplamiento / Fractura (-)
Res 95   | Res 128  |    0.00014 | Acoplamiento anómalo de estrés (+)
Res 185  | Res 128  |    0.00013 | Acoplamiento anómalo de estrés (+)
Re

## 4.2.2. Propagación distal del recableado alostérico

In [ ]:
# 1. Definimos los archivos
wt_file = "LongRange_Allostery_WT.csv"
mut_file = "LongRange_Allostery_I150A.csv"

print("Iniciando escrutinio de propagación alostérica de largo alcance (>15 Å)...")

if os.path.exists(wt_file) and os.path.exists(mut_file):
    df_wt = pd.read_csv(wt_file)
    df_mut = pd.read_csv(mut_file)

    # Adaptabilidad a los nombres de las columnas de tu script
    col_source = "Source_PDB" if "Source_PDB" in df_wt.columns else df_wt.columns[0]
    col_target = "Target_PDB" if "Target_PDB" in df_wt.columns else df_wt.columns[2]
    col_weight = "Weight" if "Weight" in df_wt.columns else "Epistatic_Coupling_JSD"

    # 2. Fusión de los grafos espaciales (alineamos por origen y destino)
    df_merged = pd.merge(df_wt, df_mut, on=[col_source, col_target], suffixes=('_WT', '_Mut'))

    # 3. Filtro Diferencial: Calculamos el Delta JSD (Mutante - WT)
    df_merged['Delta_JSD'] = df_merged[f'{col_weight}_Mut'] - df_merged[f'{col_weight}_WT']

    # 4. Filtro de Consolidación: Solo queremos rutas que AUMENTAN su transferencia
    # y las ordenamos de mayor a menor incremento.
    df_sorted = df_merged[df_merged['Delta_JSD'] > 0].sort_values(by='Delta_JSD', ascending=False)

    print("\n" + "="*70)
    print("TOP 5: RUTAS ALOSTÉRICAS EMERGENTES O FORTALECIDAS")
    print("="*70)

    # Mostrar resultados limpios
    for index, row in df_sorted.head(5).iterrows():
        source = int(row[col_source])
        target = int(row[col_target])

        # Buscar si la distancia se mantuvo en una columna específica tras el merge
        dist_col = 'Distance_WT' if 'Distance_WT' in df_merged.columns else ('Distance_Angstroms_WT' if 'Distance_Angstroms_WT' in df_merged.columns else [c for c in df_merged.columns if 'Dist' in c][0])
        dist = row[dist_col]

        wt_val = row[f'{col_weight}_WT']
        mut_val = row[f'{col_weight}_Mut']
        delta = row['Delta_JSD']

        print(f"Ruta: {source} ---> {target} | Distancia: {dist:.2f} Å | Delta: +{delta:.5f} (WT: {wt_val:.5f} -> Mut: {mut_val:.5f})")

else:
    print("\n[!] ERROR: No se encuentran los archivos LongRange_Allostery_WT.csv y LongRange_Allostery_I150A.csv")

Iniciando escrutinio de propagación alostérica de largo alcance (>15 Å)...

TOP 5: RUTAS ALOSTÉRICAS EMERGENTES O FORTALECIDAS
Ruta: 142 ---> 157 | Distancia: 23.18 Å | Delta: +0.00380 (WT: 0.01553 -> Mut: 0.01933)
Ruta: 142 ---> 156 | Distancia: 22.00 Å | Delta: +0.00368 (WT: 0.00250 -> Mut: 0.00618)
Ruta: 106 ---> 113 | Distancia: 21.40 Å | Delta: +0.00313 (WT: 0.01320 -> Mut: 0.01634)
Ruta: 45 ---> 52 | Distancia: 18.29 Å | Delta: +0.00202 (WT: 0.01372 -> Mut: 0.01574)
Ruta: 190 ---> 205 | Distancia: 18.95 Å | Delta: +0.00175 (WT: 0.00831 -> Mut: 0.01006)
